# Test Following Human with BT

In [1]:
import rclpy
from rclpy.node import Node
from tf2_ros.transform_listener import TransformListener
from tf2_ros.buffer import Buffer

from turtlebot4_navigation.turtlebot4_navigator import TurtleBot4Directions, TurtleBot4Navigator
from nav_msgs.msg import Odometry
from geometry_msgs.msg import PoseStamped

from depthai_ros_msgs.msg import TrackDetection2DArray

import math
from nav_msgs.msg import Path

In [2]:
import py_trees
import py_trees_ros.trees
import numpy as np

!pip install transformations
from transformations import quaternion_from_euler, euler_from_quaternion
from geometry_msgs.msg import Quaternion

from tf2_ros import TransformBroadcaster
from geometry_msgs.msg import TransformStamped

import numpy as np

import tf2_geometry_msgs
from geometry_msgs.msg import PointStamped, Vector3
from copy import copy
from threading import Lock


class PersonEKF:

    def __init__(self):
        self.x = np.zeros((4,1))  # [x, y, vx, vy]
        self.P = np.eye(4) * 1.0

        self.Q = np.eye(4) * 0.05   # process noise
        self.R = np.eye(2) * 0.1    # measurement noise

    def predict(self, dt):

        F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1,  0],
            [0, 0, 0,  1]
        ])

        self.x = F @ self.x
        self.P = F @ self.P @ F.T + self.Q

    def update(self, z):

        H = np.array([
            [1,0,0,0],
            [0,1,0,0]
        ])

        y = z - H @ self.x
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        I = np.eye(4)
        self.P = (I - K @ H) @ self.P
        
def euler_to_quaternion(roll: float=0, pitch: float=0, yaw: float=0) -> Quaternion:
    q = quaternion_from_euler(roll, pitch, yaw)

    quat = Quaternion()
    quat.x = q[0]
    quat.y = q[1]
    quat.z = q[2]
    quat.w = q[3]
    return quat

def quaternion_to_euler(quad: Quaternion):
    """
    Convert quaternion to Euler angles (roll, pitch, yaw)
    using ROS XYZ convention.
    """
    roll, pitch, yaw = euler_from_quaternion([quad.x, quad.y, quad.z, quad.w])
    return roll, pitch, yaw


def compute_goal_pose(robot_pose, human_pose, distance=1.0):
    """
    robot_pose: geometry_msgs/Pose
    human_pose: geometry_msgs/Pose
    distance:   desired distance from human (meters)
    """

    xr = robot_pose.position.x
    yr = robot_pose.position.y

    xh = human_pose.position.x
    yh = human_pose.position.y

    # Vector robot -> human
    dx = xh - xr
    dy = yh - yr

    d = math.hypot(dx, dy)

    if d < 1e-6:
        return None  # avoid divide-by-zero

    # Unit vector
    ux = dx / d
    uy = dy / d

    # Goal position (1m before human)
    goal_x = xh - ux * distance
    goal_y = yh - uy * distance

    # Face the human
    yaw = math.atan2(yh - goal_y, xh - goal_x)
    qx, qy, qz, qw = quaternion_from_euler(yaw, 0, 0)

    goal = PoseStamped()
    goal.header.frame_id = "odom"
    goal.pose.position.x = goal_x
    goal.pose.position.y = goal_y
    goal.pose.position.z = 0.0

    goal.pose.orientation.x = qx
    goal.pose.orientation.y = qy
    goal.pose.orientation.z = qz
    goal.pose.orientation.w = qw

    return goal

def prepare_tf_msg(stamp, trans):
    t = TransformStamped()

    t.header.stamp = stamp
    
    t.child_frame_id =  'detected_person_ekf'
    t.header.frame_id = 'map'
    
    t.transform.translation = trans

    return t

    
rclpy.init()

In [3]:
import rclpy
import py_trees
import py_trees_ros

from nav2_msgs.action import NavigateToPose

from irobot_create_msgs.action import RotateAngle
from rclpy.action import ActionClient
import time


class RelativePoseToBlackboard(py_trees.behaviour.Behaviour):
    def __init__(self, name="Pose To Blackboard", topic_name="/human_relative"):
        super(RelativePoseToBlackboard, self).__init__(name)
        self.topic_name = topic_name
        self.node = None
        self.subscriber = None
        self.msg_data = None
        
        # Initialize Blackboard
        self.blackboard = self.attach_blackboard_client()
        self.blackboard.register_key(key="human_distance", access=py_trees.common.Access.WRITE)
        self.blackboard.register_key(key="human_angle", access=py_trees.common.Access.WRITE)

    def setup(self, **kwargs):
        """ Initialize the ROS2 subscriber here """
        try:
            self.node = kwargs.get('node')
        except KeyError:
            self.feedback_message = "No ROS2 node found in setup kwargs"
            return False

        self.subscriber = self.node.create_subscription(
            Vector3,
            self.topic_name,
            self.callback,
            10
        )
        return True

    def callback(self, msg):
        # Store the message temporarily
        self.msg_data = msg

    def update(self):
        """ 
        Each tick, we check if we received a message.
        If yes, write to blackboard and return SUCCESS.
        """
        if self.msg_data:
            self.blackboard.human_distance = self.msg_data.x
            self.blackboard.human_angle = self.msg_data.y
            
            # self.feedback_message = f"Updated pose: {self.msg_data}"
            # Optional: clear msg_data if you only want to react to "new" data per tick
            # self.msg_data = None
            return py_trees.common.Status.SUCCESS
        
        self.feedback_message = "Waiting for pose data..."
        return py_trees.common.Status.RUNNING

    def terminate(self, new_status):
        self.logger.debug(f"Terminated with status: {new_status}")


class GoalToBlackboard(py_trees.behaviour.Behaviour):
    def __init__(self, name="Goal To Blackboard", topic_name="/follow_goal"):
        super(GoalToBlackboard, self).__init__(name)
        self.topic_name = topic_name
        self.node = None
        self.subscriber = None
        self.msg_data = None
        
        # Initialize Blackboard
        self.blackboard = self.attach_blackboard_client()
        self.blackboard.register_key(key="follow_human_goal", access=py_trees.common.Access.WRITE)

    def setup(self, **kwargs):
        """ Initialize the ROS2 subscriber here """
        try:
            self.node = kwargs.get('node')
        except KeyError:
            self.feedback_message = "No ROS2 node found in setup kwargs"
            return False

        self.subscriber = self.node.create_subscription(
            PoseStamped,
            self.topic_name,
            self.callback,
            10
        )
        return True

    def callback(self, msg):
        # Store the message temporarily
        self.msg_data = msg

    def update(self):
        """ 
        Each tick, we check if we received a message.
        If yes, write to blackboard and return SUCCESS.
        """
        if self.msg_data:
            self.blackboard.follow_human_goal = self.msg_data
            
            # self.feedback_message = f"Updated pose: {self.msg_data}"
            # Optional: clear msg_data if you only want to react to "new" data per tick
            # self.msg_data = None
            return py_trees.common.Status.SUCCESS
        
        self.feedback_message = "Waiting for pose data..."
        return py_trees.common.Status.RUNNING

    def terminate(self, new_status):
        self.logger.debug(f"Terminated with status: {new_status}")
        

class FollowPerson(py_trees.behaviour.Behaviour):
    def __init__(self, name="Follow Person", goal_pose_bb_name="human_pose"):
        super(FollowPerson, self).__init__(name)
        self.goal_pose_bb_name = goal_pose_bb_name
        self.blackboard = self.attach_blackboard_client()
        
        self.blackboard.register_key(key=self.goal_pose_bb_name, access=py_trees.common.Access.READ)
        
        self.navigator = None
        self.node = None
        self.current_goal_pose = None

    def setup(self, **kwargs):
        """Initialize the Navigator and ROS 2 node."""
        try:
            self.node = kwargs['node']
        except KeyError as e:
            self.feedback_message = "No ROS2 node found in setup kwargs"
            return False

        self.navigator = TurtleBot4Navigator()
        return True

    def initialise(self):
        """Called the first time the behavior is ticked."""
        self.logger.info("Initializing Follow Task")
        self.current_goal_pose = None

    def update(self) -> py_trees.common.Status:
        """The 0.5s Tick logic."""
        # 1. Fetch data from Blackboard
        if not self.blackboard.exists(self.goal_pose_bb_name):
            self.feedback_message = "No goal pose on blackboard"
            return py_trees.common.Status.FAILURE

        goal_pose = self.blackboard.get(self.goal_pose_bb_name)

        # 2. Check Freshness (0.5s Timeout)
        now = self.node.get_clock().now()
        msg_time = rclpy.time.Time.from_msg(goal_pose.header.stamp)
        age = (now - msg_time).nanoseconds / 1e9

        if age > 0.5:
            self.feedback_message = f"Data stale ({age:.2f}s). Stopping robot."
            self.stop_navigation()
            return py_trees.common.Status.RUNNING

        # 3. Decision: Should we update the Navigator goal?
        # We only send a new goal if the person moved significantly (> 0.3m)
        if self.should_update_goal(goal_pose):
            self.logger.info("Sending fresh goal to Nav2")
            self.navigator.startToPose(goal_pose)
            self.current_goal_pose = goal_pose

        # 4. Check Navigation Progress
        if self.navigator.isTaskComplete():
            nav_result = self.navigator.getResult()
            if nav_result == 4: # Succeeded
                return py_trees.common.Status.SUCCESS
            else:
                self.feedback_message = "Navigation failed or was canceled"
                return py_trees.common.Status.FAILURE

        return py_trees.common.Status.RUNNING

    def should_update_goal(self, new_pose, distance_threshold=0.3):
        """Check if the human has moved enough to warrant a new path calculation."""
        if self.current_goal_pose is None:
            return True
        
        dx = new_pose.pose.position.x - self.current_goal_pose.pose.position.x
        dy = new_pose.pose.position.y - self.current_goal_pose.pose.position.y
        distance = np.sqrt(dx**2 + dy**2)
        
        return distance > distance_threshold

    def stop_navigation(self):
        """Cancels the current Nav2 task if one is active."""
        if self.navigator and not self.navigator.isTaskComplete():
            self.navigator.cancelTask()
            self.current_goal_pose = None

    def terminate(self, new_status):
        """Clean up when the behavior stops or is interrupted."""
        if new_status == py_trees.common.Status.INVALID:
            self.stop_navigation()
        self.logger.info(f"Terminated with status: {new_status}")

In [4]:


class RotateOrStop(py_trees.behaviour.Behaviour):
    def __init__(self, name="Rotate Action Client", angle=1.57): # Default 90 deg
        super(RotateOrStop, self).__init__(name)
        self.angle = angle
        self.node = None
        self.action_client = None
        self.goal_handle = None
        self.blackboard = self.attach_blackboard_client()
        # We need read access to check the pose and a timestamp
        self.blackboard.register_key(key="human_angle", access=py_trees.common.Access.READ)
        self.last_msg_time = time.time()

    def setup(self, **kwargs):
        self.node = kwargs.get('node')
        self.action_client = ActionClient(self.node, RotateAngle, 'rotate_angle')
        return True

    def initialise(self):
        """ Reset state and send the goal """
        self.logger.info("Sending Rotate Goal")
        self.goal_handle = None
        
        # Check if we even have data to start with
        if not self.blackboard.exists("human_angle"):
            self.logger.error("No pose on blackboard!")
            return

        # angle_sign = 1 if self.blackboard.human_angle > 0 else -1
        # goal_msg = RotateAngle.Goal()
        # goal_msg.angle = self.angle * angle_sign
        # goal_msg.max_rotation_speed = 0.8

        goal_msg = RotateAngle.Goal()
        goal_msg.angle = self.blackboard.human_angle
        goal_msg.max_rotation_speed = 0.8
        
        # Note: TurtleBot4 uses radians/sec for max_rotation_speed
        self.action_client.wait_for_server()
        self._send_goal_future = self.action_client.send_goal_async(goal_msg)
        self._send_goal_future.add_done_callback(self.goal_response_callback)

    def goal_response_callback(self, future):
        self.goal_handle = future.result()

    def update(self):
        """
        The 'Tick' - Checks every ~0.5s (depending on tree tick rate)
        """
        # 1. Check message freshness
        # We assume your subscriber behavior updates 'human_pose' on the blackboard
        current_time = self.node.get_clock().now().to_msg().sec
        
        # Logic: If no human_pose exists or it hasn't been updated in 0.5s
        # You might need a 'pose_timestamp' key on the blackboard for high precision
        if not self.blackboard.exists("human_angle"):
            self.cancel_goal()
            self.feedback_message = "human_angle does not exist in Blackboard"
            return py_trees.common.Status.FAILURE
        
        if abs(self.blackboard.human_angle) < 0.34:
            self.cancel_goal()
            self.feedback_message = "human_angle is less than 0.34 rad"
            return py_trees.common.Status.FAILURE

        # 2. Action State Machine
        if not self.goal_handle:
            return py_trees.common.Status.RUNNING

        if self.goal_handle.status == 4: # Succeeded (Action Status Enum)
            return py_trees.common.Status.SUCCESS
        
        if self.goal_handle.status in [5, 6]: # Aborted or Canceled
            self.feedback_message = "goal status is Aborted or Canceled"
            return py_trees.common.Status.FAILURE

        return py_trees.common.Status.RUNNING

    def cancel_goal(self):
        if self.goal_handle:
            self.logger.info("Stopping spin: No human detected!")
            self.goal_handle.cancel_goal_async()

    def terminate(self, new_status):
        if new_status == py_trees.common.Status.INVALID:
            self.cancel_goal()
        # self.action_client = None

In [ ]:
root = py_trees.composites.Parallel(
        name="Follow Human",
        policy=py_trees.common.ParallelPolicy.SuccessOnAll(
            synchronise=False
        )
    )

topics2bb = py_trees.composites.Sequence(name="Tf2BB", memory=True)

person_relative_2bb = RelativePoseToBlackboard(name="Relative Pose to BlackBoard")
goal_pose_2bb = GoalToBlackboard(name="Goal to BlackBoard")

tasks = py_trees.composites.Selector("Tasks", memory=False)

def check_distance(blackboard: py_trees.blackboard.Blackboard):
    try:
        human_distance = blackboard.human_distance
    except KeyError:
        return False
        
    if human_distance is None:
        return False
    else:
        return human_distance > 1.5

human_closed = py_trees.decorators.EternalGuard(
    name="Human is further than 1m?",
    condition=check_distance,
    blackboard_keys={"human_distance": None},
    child=FollowPerson(
        name="Come to human",
        goal_pose_bb_name="follow_human_goal",
        # human_relative_bb_name="human_relative"
    )
)

def check_angle(blackboard: py_trees.blackboard.Blackboard):
    try:
        human_angle = blackboard.human_angle
    except KeyError:
        return False
        
    if human_angle is None:
        return False
    else:
        return abs(human_angle) > 0.1

human_center = py_trees.decorators.EternalGuard(
    name="Human is out of center?",
    condition=check_angle,
    blackboard_keys={"human_angle": None},
    child=RotateOrStop(
        name="RotateToHuman",
        angle=0.8
    )
)

idle = py_trees.behaviours.Running(name="Idle")

root.add_child(topics2bb)
topics2bb.add_children([
    person_relative_2bb,
    goal_pose_2bb
])

root.add_child(tasks)
tasks.add_children([
    human_center,
    human_closed, 
    idle])

tree = py_trees_ros.trees.BehaviourTree(
        root=root,
        unicode_tree_debug=True
    )

try:
    tree.setup(timeout=15)
except py_trees_ros.exceptions.TimedOutError as e:
    console.logerror(console.red + "failed to setup the tree, aborting [{}]".format(str(e)) + console.reset)
    tree.shutdown()

tree.tick_tock(period_ms=50.0)

try:
    rclpy.spin(tree.node)
except (KeyboardInterrupt, rclpy.executors.ExternalShutdownException):
    pass
finally:
    tree.shutdown()
    rclpy.try_shutdown()


/_/ Follow Human [*]
    {-} Tf2BB [*]
        --> Relative Pose to BlackBoard [*] -- Waiting for pose data...
        --> Goal to BlackBoard
    [o] Tasks [*]
        -^- Human is out of center? [✕]
            --> RotateToHuman
        -^- Human is further than 1m? [✕]
            --> Come to human
        --> Idle [*] -- running


/_/ Follow Human [*]
    {-} Tf2BB [*]
        --> Relative Pose to BlackBoard [*] -- Waiting for pose data...
        --> Goal to BlackBoard
    [o] Tasks [*]
        -^- Human is out of center? [✕]
            --> RotateToHuman
        -^- Human is further than 1m? [✕]
            --> Come to human
        --> Idle [*] -- running


/_/ Follow Human [*]
    {-} Tf2BB [*]
        --> Relative Pose to BlackBoard [*] -- Waiting for pose data...
        --> Goal to BlackBoard
    [o] Tasks [*]
        -^- Human is out of center? [✕]
            --> RotateToHuman
        -^- Human is further than 1m? [✕]
            --> Come to human
        --> Idle [*] -- r

[INFO] [1771919587.333213834] [basic_navigator]: Navigating to goal: -0.3457155000731633 1.388055746410619...


[ INFO] Come to human        : Terminated with status: Status.FAILURE

/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [✕]
            --> RotateToHuman [✕] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m? [✕]
            --> Come to human [✕] -- Navigation failed or was canceled
        --> Idle [*] -- running

[ INFO] RotateToHuman        : Sending Rotate Goal
[ INFO] Come to human        : Initializing Follow Task

/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [✕]
            --> RotateToHuman [✕] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m? [*]
            --> Come to human [*] -- Data stale (3.80s). Stopping robot.
        --

[INFO] [1771919590.745687372] [basic_navigator]: Goal succeeded!



/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [*]
            --> RotateToHuman [*] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m?
            --> Come to human
        --> Idle


/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [*]
            --> RotateToHuman [*] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m?
            --> Come to human
        --> Idle


/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [*]
            --> RotateToHuman [*] -- human_angle is less than 0.34 rad


...

[ INFO] Come to human        : Terminated with status: Status.FAILURE

/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [✕]
            --> RotateToHuman [✕] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m? [✕]
            --> Come to human [✕] -- Navigation failed or was canceled
        --> Idle [*] -- running

[ INFO] RotateToHuman        : Sending Rotate Goal
[ INFO] Come to human        : Initializing Follow Task

/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [✕]
            --> RotateToHuman [✕] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m? [*]
            --> Come to human [*] -- Data stale (5.29s). Stopping robot.
        --

[INFO] [1771919686.460837833] [basic_navigator]: Goal succeeded!



/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [*]
            --> RotateToHuman [*] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m?
            --> Come to human
        --> Idle


/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [*]
            --> RotateToHuman [*] -- human_angle is less than 0.34 rad
        -^- Human is further than 1m?
            --> Come to human
        --> Idle


/_/ Follow Human [*]
    {-} Tf2BB [✓]
        --> Relative Pose to BlackBoard [✓] -- Waiting for pose data...
        --> Goal to BlackBoard [✓]
    [o] Tasks [*]
        -^- Human is out of center? [*]
            --> RotateToHuman [*] -- human_angle is less than 0.34 rad


[INFO] [1771919815.338432286] [basic_navigator]: Navigating to goal: -1.7401506346589062 -2.305910965119042...


Estimated time of arrival: 0seconds.            

In [ ]:
(388062895 - 515532193) / 1e9